# Deforestation v2: full run on Colab
Runtime → Change runtime type → **T4 GPU**. Results are saved to Google Drive, so a disconnect only loses the run in progress. Re-run all cells to resume.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/deforestation-v2'
!mkdir -p $WORK

In [ ]:
%cd /content
!rm -rf deforestation-prediction-india
!git clone -q https://github.com/kavya-sajjit/deforestation-prediction-india.git
%cd deforestation-prediction-india
!git checkout -q v2-forecast-bcos 2>/dev/null || echo 'already on main'
!pip install -q -r requirements.txt

## 1. Tests (synthetic data, offline)

In [ ]:
!pytest -q tests/

## 2. Build the dataset (Sentinel-2 + Hansen). About 10–25 min per region.

In [ ]:
import os
if not os.path.exists(f'{WORK}/data/v2/dataset_summary.json'):
    !python -m deforest.data.build --out $WORK/data/v2
!cat $WORK/data/v2/dataset_summary.json

Check the class counts above. If a region has fewer than ~40 forecast positives, move its box with `scripts/scan_hansen_aoi.py`, edit `deforest/config.py`, delete that region's files and rebuild it with `--regions <name>`.

## 3. Spatial 5-fold CV (the main table)

In [ ]:
!python -m deforest.experiments --data $WORK/data/v2 --out $WORK/results --profile core --scheme block

## 4. Leave-one-region-out

In [ ]:
!python -m deforest.experiments --data $WORK/data/v2 --out $WORK/results --profile core --scheme loro --only history_lr gbm cnn_rgbn_hist bcos_rgbn_hist bcos_rgbn cnn_bitemporal bcos_bitemporal

## 5. Risk maps (out-of-region models) and summary

In [ ]:
for r in ['western_ghats','odisha_mining','northeast_hills','central_india']:
    !python -m deforest.risk_map --data $WORK/data/v2 --region {r} --model $WORK/results/forecast/loro/bcos_rgbn_hist/test_{r}/model.pt --out $WORK/maps

In [ ]:
!python -m deforest.summarize $WORK/results
!rm -rf results maps && cp -r $WORK/results results && cp -r $WORK/maps maps
!python scripts/update_readme.py

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/SUMMARY.md').read()))
import glob
for f in sorted(glob.glob('maps/*.png'))+sorted(glob.glob('results/*/block/*/fold0/gallery.png')):
    print(f); display(Image(f))

## 6. Commit the results
Download `results/` (without `model.pt` files if they are large), `maps/*.png` and `README.md`, and commit them; or push from here with a GitHub token.